# Grid-to-Catchment Interpolation Weight Computation

Computes area-weighted interpolation weights that map a gridded dataset onto catchment polygons.
The resulting weight DataFrame can then be used to aggregate any gridded time series into
basin-averaged values via a weighted sum.

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
from shapely.geometry import box
from joblib import Parallel, delayed
from tqdm import tqdm

## Configuration

Set paths and parallelism below.

In [2]:
BASINS_PATH  = "./data/basins.pkl"       # GeoSeries of catchment polygons
NETCDF_PATH  = "/data_prediction005/SYSTEM/prediction002/home/tristan/data/ECMWF/era/eraL_1993-02.nc"  # any ERA5 file (used only for the grid)
OUTPUT_PATH  = "./data/era_weights.pkl"  # where to save the weight DataFrame
N_JOBS       = -1                        # number of parallel workers (-1 = all CPUs)

## Load Data

In [3]:
# Basin polygons – a GeoSeries indexed by basin IDs
basins = pd.read_pickle(BASINS_PATH)
print(f"Loaded {len(basins)} basin polygons")
print(f"CRS      : {basins.crs}")
print(f"Bounds   : {basins.total_bounds}  (minx, miny, maxx, maxy)")
basins.head(3)

Loaded 8893 basin polygons
CRS      : None
Bounds   : [129.62222222  30.29       146.235       45.45055556]  (minx, miny, maxx, maxy)


2600468634    POLYGON ((134.99889 34.82667, 134.99861 34.826...
2336227590    MULTIPOLYGON (((140.09472 36.02361, 140.09472 ...
2336228014    POLYGON ((140.15 36.06056, 140.15 36.06028, 14...
dtype: geometry

In [4]:
# ERA5 grid – load one file just to extract coordinate arrays
ds = xr.open_dataset(NETCDF_PATH)
print(ds)

grid_lat = ds.latitude.values   # must be monotonically decreasing
grid_lon = ds.longitude.values  # must be monotonically increasing

print(f"\nLatitude  : {grid_lat[0]:.2f} → {grid_lat[-1]:.2f}  ({len(grid_lat)} pts, step {grid_lat[1]-grid_lat[0]:.3f})")
print(f"Longitude : {grid_lon[0]:.2f} → {grid_lon[-1]:.2f}  ({len(grid_lon)} pts, step {grid_lon[1]-grid_lon[0]:.3f})")

assert np.all(np.diff(grid_lat) <= 0), "grid_lat must be monotonically decreasing"
assert np.all(np.diff(grid_lon) >= 0), "grid_lon must be monotonically increasing"

<xarray.Dataset> Size: 2GB
Dimensions:     (valid_time: 672, latitude: 201, longitude: 301)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 5kB 1993-02-01 ... 1993-02-28T23:...
    expver      (valid_time) <U4 11kB ...
  * latitude    (latitude) float64 2kB 50.0 49.9 49.8 49.7 ... 30.2 30.1 30.0
  * longitude   (longitude) float64 2kB 120.0 120.1 120.2 ... 149.8 149.9 150.0
    number      int64 8B ...
Data variables: (12/14)
    u10         (valid_time, latitude, longitude) float32 163MB ...
    v10         (valid_time, latitude, longitude) float32 163MB ...
    d2m         (valid_time, latitude, longitude) float32 163MB ...
    t2m         (valid_time, latitude, longitude) float32 163MB ...
    pev         (valid_time, latitude, longitude) float32 163MB ...
    sd          (valid_time, latitude, longitude) float32 163MB ...
    ...          ...
    sp          (valid_time, latitude, longitude) float32 163MB ...
    tp          (valid_time, latitude, longitude) float32 163MB 

## Build Grid Mask

To avoid computing polygon–cell intersections for the entire global (or regional) grid,
we restrict computation to grid **cells** whose bounding box overlaps the total extent
of all basin polygons.

Grid cell `(i, j)` is the box bounded by consecutive coordinate values
`[grid_lon[j], grid_lon[j+1]] × [grid_lat[i+1], grid_lat[i]]`.
Indices therefore run from `0` to `n-2` (one fewer than the number of coordinate points).

In [5]:
# Cell-edge arrays (length n-1)
cell_lat_min = np.minimum(grid_lat[:-1], grid_lat[1:])  # southward edge of each cell row
cell_lat_max = np.maximum(grid_lat[:-1], grid_lat[1:])  # northward edge
cell_lon_min = np.minimum(grid_lon[:-1], grid_lon[1:])  # western edge of each cell column
cell_lon_max = np.maximum(grid_lon[:-1], grid_lon[1:])  # eastern edge

# Overall polygon bounding box with one-cell padding
minx, miny, maxx, maxy = basins.total_bounds
pad = max(grid_lat[0] - grid_lat[1], grid_lon[1] - grid_lon[0])  # one grid step

lat_cov = (cell_lat_max >= miny - pad) & (cell_lat_min <= maxy + pad)
lon_cov = (cell_lon_max >= minx - pad) & (cell_lon_min <= maxx + pad)

# 2-D boolean mask over cell indices, then convert to (row_indices, col_indices)
msk = np.where(np.outer(lat_cov, lon_cov))

print(f"Active cells in mask : {len(msk[0])}  "
      f"(out of {len(grid_lat)-1} × {len(grid_lon)-1} = {(len(grid_lat)-1)*(len(grid_lon)-1)} total)")

Active cells in mask : 26195  (out of 200 × 300 = 60000 total)


## Interpolation Weight Functions

In [6]:
def _closest_cell(poly, grid_lat, grid_lon, msk):
    """Fallback: assign weight to the single grid cell whose center is nearest to the polygon centroid."""
    cy, cx = poly.centroid.y, poly.centroid.x
    # Cell centers are midpoints between adjacent coordinate values
    cell_lat_ctr = (grid_lat[:-1] + grid_lat[1:]) / 2
    cell_lon_ctr = (grid_lon[:-1] + grid_lon[1:]) / 2
    coords = np.stack([cell_lat_ctr[msk[0]], cell_lon_ctr[msk[1]]], axis=1)
    idx = np.argmin(((np.array([[cy, cx]]) - coords) ** 2).sum(axis=1))
    i, j = int(msk[0][idx]), int(msk[1][idx])
    return [(i, j)], [poly.area]


def _polygon_weights(polygon, grid_lat, grid_lon, msk,
                     cell_lat_min, cell_lat_max, cell_lon_min, cell_lon_max):
    """
    Compute intersection-area weights between *polygon* and all grid cells in *msk*.

    Parameters
    ----------
    polygon : shapely geometry
    grid_lat, grid_lon : 1-D arrays of coordinate values
    msk : tuple (row_indices, col_indices) into the cell arrays (length n-1 each)
    cell_lat_min/max, cell_lon_min/max : pre-computed cell-edge arrays (length n-1)

    Returns
    -------
    grid_indices : list of (i, j) cell index pairs
    weights      : list of intersection areas (same length)
    """
    minx, miny, maxx, maxy = polygon.bounds

    # Restrict candidates to cells whose bounding box overlaps the polygon bbox
    lat_ok = (cell_lat_max[msk[0]] >= miny) & (cell_lat_min[msk[0]] <= maxy)
    lon_ok = (cell_lon_max[msk[1]] >= minx) & (cell_lon_min[msk[1]] <= maxx)
    candidates = np.where(lat_ok & lon_ok)[0]

    grid_indices, weights = [], []
    for k in candidates:
        i, j = int(msk[0][k]), int(msk[1][k])
        cell_box = box(cell_lon_min[j], cell_lat_min[i], cell_lon_max[j], cell_lat_max[i])
        intersection = polygon.intersection(cell_box)
        if not intersection.is_empty:
            grid_indices.append((i, j))
            weights.append(intersection.area)

    if weights:
        return grid_indices, weights
    # No overlap found (tiny polygon between grid cells) → fall back to nearest cell
    return _closest_cell(polygon, grid_lat, grid_lon, msk)


def _worker(polygon_id, polygon, grid_lat, grid_lon, msk,
            cell_lat_min, cell_lat_max, cell_lon_min, cell_lon_max):
    grid_indices, weights = _polygon_weights(
        polygon, grid_lat, grid_lon, msk,
        cell_lat_min, cell_lat_max, cell_lon_min, cell_lon_max
    )
    return [
        {"polygon_index": polygon_id, "i": i, "j": j, "weight": w}
        for (i, j), w in zip(grid_indices, weights)
    ]


def compute_weights(geoseries, grid_lat, grid_lon, msk, n_jobs=None):
    """
    Compute normalised area-weighted interpolation weights for every polygon.

    Parameters
    ----------
    geoseries : GeoSeries
        Catchment polygons; index values become the 'polygon_index' column.
    grid_lat, grid_lon : 1-D arrays
        Coordinate values of the grid (lat decreasing, lon increasing).
    msk : tuple
        (row_indices, col_indices) of active cells, e.g. from np.where(mask_2d).
        Indices are into the cell-edge arrays (length n-1).
    n_jobs : int, optional
        Passed to joblib.Parallel. Default None (sequential).

    Returns
    -------
    DataFrame with columns: polygon_index, i, j, weight, norm_weight
        norm_weight sums to 1.0 for each polygon.
    """
    assert np.all(np.diff(grid_lat) <= 0), "grid_lat must be decreasing"
    assert np.all(np.diff(grid_lon) >= 0), "grid_lon must be increasing"

    # Pre-compute cell edges once (avoids recomputing inside each worker)
    cell_lat_min = np.minimum(grid_lat[:-1], grid_lat[1:])
    cell_lat_max = np.maximum(grid_lat[:-1], grid_lat[1:])
    cell_lon_min = np.minimum(grid_lon[:-1], grid_lon[1:])
    cell_lon_max = np.maximum(grid_lon[:-1], grid_lon[1:])

    results = Parallel(n_jobs=n_jobs)(
        delayed(_worker)(
            idx, poly, grid_lat, grid_lon, msk,
            cell_lat_min, cell_lat_max, cell_lon_min, cell_lon_max
        )
        for idx, poly in tqdm(geoseries.items(), total=len(geoseries), desc="Computing weights")
    )

    records = [row for polygon_rows in results for row in polygon_rows]
    df = pd.DataFrame(records)

    poly_area = df.groupby("polygon_index")["weight"].transform("sum")
    df["norm_weight"] = df["weight"] / poly_area

    return df

## Compute Weights

In [7]:
weights_df = compute_weights(basins, grid_lat, grid_lon, msk, n_jobs=N_JOBS)
print(f"\nResult shape : {weights_df.shape}")
weights_df.head(10)

Computing weights: 100%|██████████| 8893/8893 [00:18<00:00, 474.65it/s]



Result shape : (28084, 5)


,polygon_index,i,j,weight,norm_weight
0,2600468634,151,149,0.000007,0.001603
1,2600468634,151,150,0.004083,0.975197
2,2600468634,151,151,0.000097,0.023200
3,2336227590,139,200,0.000045,0.021059
4,2336227590,139,201,0.000538,0.252207
5,2336227590,140,201,0.001550,0.726733
6,2336228014,139,201,0.001948,0.355164
7,2336228014,139,202,0.000756,0.137846
8,2336228014,140,201,0.000477,0.086955
9,2336228014,140,202,0.002304,0.420035


## Validate

Normalised weights must sum exactly to 1 for every polygon.

In [8]:
norm_sums = weights_df.groupby("polygon_index")["norm_weight"].sum()
max_err = (norm_sums - 1).abs().max()
print(f"Max deviation from 1.0 : {max_err:.2e}")
assert max_err < 1e-7, "Weights do not sum to 1 — check for grid/polygon CRS mismatch"
print(f"All {len(norm_sums)} polygons validated ✓")

# Coverage stats
cells_per_basin = weights_df.groupby("polygon_index").size()
print(f"\nGrid cells per basin — min: {cells_per_basin.min()}  "
      f"median: {cells_per_basin.median():.0f}  max: {cells_per_basin.max()}")
print(f"Basins covered by a single cell (nearest-neighbour fallback): "
      f"{(cells_per_basin == 1).sum()}")

Max deviation from 1.0 : 2.22e-16
All 8893 polygons validated ✓

Grid cells per basin — min: 1  median: 3  max: 11
Basins covered by a single cell (nearest-neighbour fallback): 1039


## Save

In [9]:
weights_df.to_pickle(OUTPUT_PATH)
print(f"Saved {len(weights_df)} rows → {OUTPUT_PATH}")
print(weights_df.dtypes)

Saved 28084 rows → ./data/era_weights.pkl
polygon_index      int64
i                  int64
j                  int64
weight           float64
norm_weight      float64
dtype: object


> **Caution**: you may need to treat data units carefully, and rename some columns for this dataframe to work with the diffhydro catchment interpolator. We can do this together next.